# Google Hubbard conversion audit

## TL;DR
Published local Z/ZZ values can be converted to our definitions. The models are not identical; no advantage claim follows.

## Context and methods
U=8, T=.4, no flux, holon stripe, 5x5/6x6, 2/3 layers. No parameter fitting, clipping, N substitution or interpolation. Reference differences and metadata limitations are preserved in the report.

In [1]:
from pathlib import Path
import sys, json, contextlib, io, unittest
import numpy as np
root=Path.cwd().resolve()
assert (root/'convert_google_hubbard.py').exists()
sys.path.insert(0,str(root))
from convert_google_hubbard import RUN, compute
from package_google_conversion import audit_and_compare
from test_google_conversion import ConversionTests
suite=unittest.defaultTestLoader.loadTestsFromTestCase(ConversionTests)
checks=unittest.TextTestRunner(verbosity=2).run(suite)
assert checks.wasSuccessful()

test_complete_profile (test_google_conversion.ConversionTests.test_complete_profile) ... 

ok


test_fock_states (test_google_conversion.ConversionTests.test_fock_states) ... 

ok


test_missing_duplicate_and_bounds (test_google_conversion.ConversionTests.test_missing_duplicate_and_bounds) ... 

...

test_shapes (test_google_conversion.ConversionTests.test_shapes) ... 

ok


test_spin_not_doubled (test_google_conversion.ConversionTests.test_spin_not_doubled) ... 

ok


----------------------------------------------------------------------
Ran 8 tests in 0.052s

OK


## Data
Re-read all four selected HDF5 inputs. Original fetch manifests contain URL, CRC and SHA256. This cell is local analysis only; it does not fetch data or submit jobs.

In [2]:
with contextlib.redirect_stdout(io.StringIO()):
    fresh=compute()
audit=audit_and_compare(fresh)
print('Independent RMS max delta:',audit['stdlib_RMS_check_max_abs'])
print('Chi convergence:',audit['convergence'])
print('Initial metadata records:',audit['metadata_audit']['records'], 'distinct:',audit['metadata_audit']['distinct_configurations'])
archived=audit_and_compare()
assert audit==archived
print('Fresh HDF conversion matches archived comparison exactly')

Independent RMS max delta: 5.551115123125783e-17
Chi convergence: [{'side': 5, 'chi_from': 1024, 'chi_to': 2048, 'rms': [4.175841841852513e-06, 1.4248867130214371e-05, 8.606818988162691e-06]}, {'side': 6, 'chi_from': 1024, 'chi_to': 2048, 'rms': [2.8149213782167222e-05, 0.0002133232488121019, 7.673143550079575e-05]}]
Initial metadata records: 16 distinct: 1


Fresh HDF conversion matches archived comparison exactly


## Results
Each estimate is compared only to its own model reference; these are not all exact references.

In [3]:
import pandas as pd
cols=['case','group','N','N0','mean_D','ref_D','spin_RMS','rms_charge','rms_spin','rms_D']
display(pd.DataFrame(audit['rows'])[cols])

,case,group,N,N0,mean_D,ref_D,spin_RMS,rms_charge,rms_spin,rms_D
0,Google 5x5 / 2 lagen,mitigated,20.138685,20,0.066634,0.063199,0.603074,0.017793,0.022798,0.010211
1,Google 5x5 / 2 lagen,raw,21.749105,20,0.150411,0.063199,0.371386,0.123002,0.221038,0.089746
2,Google 5x5 / 3 lagen,mitigated,20.259722,20,0.077880,0.063199,0.553725,0.024314,0.042689,0.018409
3,Google 5x5 / 3 lagen,raw,22.579590,20,0.180553,0.063199,0.278408,0.177542,0.314295,0.118747
4,Google 6x6 / 2 lagen,mitigated,30.508690,30,0.089658,0.070583,0.564105,0.028092,0.044776,0.024966
5,Google 6x6 / 2 lagen,raw,33.628135,30,0.199131,0.070583,0.254908,0.174594,0.337402,0.129749
6,Google 6x6 / 3 lagen,mitigated,30.443448,30,0.096457,0.070583,0.516458,0.026611,0.080226,0.029714
7,Google 6x6 / 3 lagen,raw,34.455530,30,0.220839,0.070583,0.165322,0.222049,0.426621,0.151228
8,Ons 6x6 / original,raw,35.463379,32,0.240655,0.087896,0.016324,0.274992,0.701476,0.157102
9,Ons 6x6 / original,readout,35.492762,32,0.240943,0.087896,0.017180,0.276104,0.701677,0.157414


## Takeaways
The 5x5 emphasis does not mean absence of quantitative 6x6 data. In this selected context Google local residuals are smaller against its own, more stable TDVP reference. Different hopping, holes, finite-circuit approximations and mitigation prohibit a matched-accuracy ranking. The external initial_states.json is inconsistent; HDF t=0 anchors show the nominal N=20/30, not raw hardware conservation. Covariance is unavailable, so aggregate confidence intervals are not invented.